# RAG Machine Learning Knowledge Assistant
End-to-end Retrieval-Augmented Generation over *Introduction to Machine Learning with Python* (Müller & Guido).

**How to run:** `pip install -r requirements.txt`, put the book at `data/intro-to-ml-book.pdf`, set `GEMINI_API_KEY` (or `OPENAI_API_KEY`) in `.env`, then *Run All*.
Set `BACKEND = "tfidf"` and `PROVIDER = "extractive"` to run fully offline with no downloads or keys (lower quality baseline).

In [ ]:
import sys, json, os
sys.path.insert(0, os.path.abspath(".."))          # make the `rag` package importable from notebooks/
from rag.config import Config
from rag.loader import load_document, explore
from rag.chunker import chunk_pages, chunk_stats
from rag.retriever import RagIndex
from rag.pipeline import RagPipeline
from rag.experiments import load_questions, evaluate_k, evaluate_chunking, page_title_map, score_profile

BACKEND  = "st"        # st (SentenceTransformers) | openai | gemini | tfidf (offline)
PROVIDER = "gemini"    # gemini | openai | ollama | extractive (offline)
cfg = Config(pdf_path="data/intro-to-ml-book.pdf", index_dir="index_nb",
             embedding_backend=BACKEND, llm_provider=PROVIDER, chunk_size=500, chunk_overlap=100).resolved()
cfg

## Part 1 - Data understanding & preprocessing
**1-2. Load the PDF and extract text from all pages** (pypdf; the PDF outline gives chapter/section for every page).

In [ ]:
pages = load_document(cfg.path(cfg.pdf_path), cfg.first_title, cfg.last_title)
print("pages loaded:", len(pages))

**3. Explore the document structure and text-quality issues**

In [ ]:
stats = explore(pages)
print(json.dumps({k: v for k, v in stats.items() if k != "chapters"}, indent=2))
print("\npages per chapter:"); [print(f"  {v:>3}  {k}") for k, v in stats["chapters"].items()]

Issues found (and handled in `rag/loader.py`): running footers (`86 | Chapter 2: ...`) on every page, words hyphenated across lines with U+2010,
a kerning artefact (`Y ou` for `You`), `tf-idf` written with an en dash, code cells (`In[n]`/`Out[n]`) that must not be re-flowed,
footers glued to code lines, 5 blank pages / chapter openers without footers, front matter and the Index (excluded).
Before/after for one page:

In [ ]:
p = pages[60]
print("---- RAW ----\n", p.raw_text[:600]); print("\n---- CLEAN ----\n", p.text[:600])
print("\nsection:", p.chapter, ">", p.section, "| printed page", p.label)

**4. Split into chunks** (500 tokens, 100 overlap; chunks never cross chapters, and carry chapter/section/page metadata).

In [ ]:
chunks = chunk_pages(pages, cfg.chunk_size, cfg.chunk_overlap)
print(chunk_stats(chunks))
c = chunks[120]; print("\n", c.citation(), "-", c.n_tokens, "tokens\n"); print(c.text[:700])

## Part 2 - Embeddings & vector database
Default: **BAAI/bge-small-en-v1.5** (384-d, normalised) stored in a **FAISS** flat inner-product index (== cosine similarity). See `REPORT.md` for the rationale.

In [ ]:
index = RagIndex.build(chunks, cfg, cfg.path(cfg.index_dir))
print(index.manifest)

In [ ]:
def show(hits, n=350):
    for h in hits:
        print(f"[{h.rank}] score={h.score:.3f}  {h.chunk.citation()}\n    " + " ".join(h.chunk.text.split())[:n] + " ...\n")
show(index.search("What is overfitting?", 3))

## Part 3 - Retrieval pipeline
Query -> embedding -> top-k. Experiment with k = 3, 5, 7, using the section-labelled evaluation set in `eval/questions.json`.

In [ ]:
qs = load_questions()
titles = page_title_map(cfg)
k_res = evaluate_k(index, qs, titles, (3, 5, 7))
import pandas as pd
pd.DataFrame({k: {m: v[m] for m in ("hit_rate", "mrr", "precision", "keyword_coverage", "avg_context_tokens")} for k, v in k_res.items()}).T

In [ ]:
# chunk size / overlap study (re-embeds the book 4 times; ~1-2 min with bge-small on CPU)
pd.DataFrame(evaluate_chunking(cfg, qs, [(300, 50), (500, 100), (800, 150), (1000, 200)], k=5))

In [ ]:
# top-1 similarity of in-scope vs out-of-scope questions -> choose the refusal threshold (min_score)
print(json.dumps(score_profile(index, qs), indent=2))

## Part 4 - Answer generation (RAG)
The prompt (see `rag/generator.py`): answer **only** from numbered CONTEXT, refuse with a fixed sentence when unsupported, cite `[n]`, never invent code or numbers; temperature 0.
A retrieval-score gate refuses *before* calling the LLM when even the best chunk is weakly related.

In [ ]:
from rag.generator import SYSTEM_PROMPT, build_user_prompt, lexical_support
print(SYSTEM_PROMPT)

In [ ]:
pipe = RagPipeline.from_dir(cfg)
r = pipe.ask("How does a random forest reduce overfitting?", k=5)
print(r.answer, "\n"); show(r.hits, 200)
print("lexical support:", lexical_support(r.answer, r.hits))

In [ ]:
# prompt that was actually sent to the LLM (evidence of grounding: numbered, source-tagged context)
print(r.prompt[:1800], "...")

In [ ]:
# grounding test: questions the book cannot answer must be refused, not hallucinated
for q in ["Who won the 2018 FIFA World Cup?", "How do transformers use self-attention in large language models?"]:
    a = pipe.ask(q); print("Q:", q, "\nA:", a.answer, "| refused:", a.refused, "\n")

## Part 5 - End-to-end application & multi-query demo
The same pipeline is served by `streamlit run app.py` (chat UI with sources) and `python -m rag.cli chat` (terminal).

In [ ]:
demo_questions = [
    "What is the difference between supervised and unsupervised learning?",
    "What is cross-validation and why is it used?",
    "How does grid search work for tuning model parameters?",
    "What does the parameter C control in logistic regression and linear SVMs?",
    "What is a pipeline in scikit-learn and why use it?",
    "What does tf-idf do?",
]
for q in demo_questions:
    a = pipe.ask(q)
    print(f"Q: {q}\nA: {a.answer}\n   sources: " + " | ".join(f"[{h.rank}] {h.chunk.citation()}" for h in a.hits) + "\n")